<a href="https://colab.research.google.com/github/swirita/salmonellosis-forecasting-analysis/blob/main/notebooks/04_salmonellosis_ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Random Forest forecasting of MMWR weekly cases

## Scenario

We want to forecast **`Current week` cases** for future MMWR weeks.

Synthetic data contains:
- 2022–2025: complete historical years
- 2026: observed through **MMWR Week 37**
- 2026 Weeks 38–52: unknown future values that we will forecast

The target is:

**`Current week`**

The model uses only information that would be available when making the forecast.


In [ ]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import os
import matplotlib.pyplot as plt


## Plot Colors

Defined once here and reused in every chart below.


In [ ]:
NAVY = "#252653"
CYAN = "#68B8C4"
LIGHT_GRAY = "#E7E9F0"
TEXT_GRAY = "#555B70"


## Helper Functions


In [ ]:
def mmwr_week_to_season(week):
    """Classify an MMWR week number into a season.

    Matches the Season feature already computed for historical weeks in
    the data-preparation notebook; used here for future weeks that don't
    have a Season value yet.
    """
    if week <= 13:
        return "Winter"
    elif week <= 26:
        return "Spring"
    elif week <= 39:
        return "Summer"
    else:
        return "Fall"


def predict_next_week(
    model, features, area, year, week,
    previous_52_max, ytd_previous_year, season, previous_week_cases,
):
    """Build one feature row for a single area/week and return the model's
    non-negative prediction.

    Shared by both the 2025 validation loop and the 2026 forecast loop,
    since both do the same thing: turn known history into one row of
    features, predict, and floor the prediction at zero.
    """
    row = pd.DataFrame([{
        "Reporting Area": area,
        "Current MMWR Year": year,
        "MMWR WEEK": week,
        "Previous 52 week Max": previous_52_max,
        "Cumulative YTD Previous MMWR Year": ytd_previous_year,
        "Season": season,
        "Previous week cases": previous_week_cases,
    }])
    prediction = float(model.predict(row[features])[0])
    return max(prediction, 0)


## Load and Prepare Data

In [ ]:
# Connect Google Colab to Google Drive
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [ ]:
# Project folders
project_folder = "/content/drive/MyDrive/Salmonellosis Forecasting and Analysis"
data_folder = os.path.join(project_folder, "data")

# Load the cleaned data created in the preparation notebook
cleaned_file_path = os.path.join(data_folder, "salmonellosis_weekly_cleaned.csv")
df = pd.read_csv(cleaned_file_path)

df = df.sort_values(
    ["Reporting Area", "Current MMWR Year", "MMWR WEEK"]
).reset_index(drop=True)

df.head()

,Reporting Area,Current MMWR Year,MMWR WEEK,Label,Current week,Previous 52 week Max,Cumulative YTD Current MMWR Year,Cumulative YTD Previous MMWR Year,Season,Previous week cases
0,ALABAMA,2022,1,Salmonellosis (excluding Salmonella Typhi infe...,10,38.0,10,6.0,Winter,2.0
1,ALABAMA,2022,2,Salmonellosis (excluding Salmonella Typhi infe...,3,38.0,18,12.0,Winter,10.0
2,ALABAMA,2022,3,Salmonellosis (excluding Salmonella Typhi infe...,0,38.0,24,17.0,Winter,3.0
3,ALABAMA,2022,4,Salmonellosis (excluding Salmonella Typhi infe...,2,38.0,26,25.0,Winter,0.0
4,ALABAMA,2022,5,Salmonellosis (excluding Salmonella Typhi infe...,0,38.0,29,30.0,Winter,2.0
